## Preparation

In [8]:
import os
import sys
parent_dir = os.path.abspath(os.path.join(os.path.join(os.getcwd(), ".."), ".."))
sys.path.append(parent_dir)
from pathlib import Path
import time
import json

import numpy as np
from data_generation.models.tech_substitution import TechnologySubstitution
from data_generation.models.tech_substitution import TechSubNumericalSolver
from data_generation.models.general_ode_solver import FitzHughNagumoModel
from data_generation.models.general_ode_solver import GeneralODENumericalSolver
from data_generation.simulations.grid import Grid, fractional_transformation, logistic_transformation
from data_generation.simulations.simulator import run_and_store_simulations

In [9]:
OUTPUT_DIR = Path.cwd().parent.parent / "datasets" / "results"
OUTPUT_DIR


PosixPath('/home/kstiller/masterthesis/Coding/mdp-world-model/datasets/results')

In [10]:
#Create the run_dict #Overwrites EVERYTHING, Take care!
'''
run_dict = dict()
run_dict['TechSub'] = dict()
run_dict['FHN-Model'] = dict()
run_dict['TechSub'][0] = None
run_dict['FHN-Model'][0] = None

# Save the run_dict to a JSON file
with open(OUTPUT_DIR / "run_dict.json", "w") as file:
    json.dump(run_dict, file, indent=4)
'''


'\nrun_dict = dict()\nrun_dict[\'TechSub\'] = dict()\nrun_dict[\'FHN-Model\'] = dict()\nrun_dict[\'TechSub\'][0] = None\nrun_dict[\'FHN-Model\'][0] = None\n\n# Save the run_dict to a JSON file\nwith open(OUTPUT_DIR / "run_dict.json", "w") as file:\n    json.dump(run_dict, file, indent=4)\n'

In [11]:
# Open the JSON file again and read its content
with open(OUTPUT_DIR / "run_dict.json", "r") as file:
    run_dict = json.load(file)


In [12]:
def update_run_dict(model_run_dict, num_samples_per_cell, resolution, controls, transformations, trafo_params, delta_t, comment = None):
    next_key = sorted([int(key) for key in model_run_dict.keys()])[-1] +1 
    model_run_dict[next_key] = dict()
    model_run_dict[next_key]['run_ids'] = list()
    model_run_dict[next_key]['resolution'] = resolution
    model_run_dict[next_key]['num_samples_per_cell'] = num_samples_per_cell
    model_run_dict[next_key]['controls'] = list(controls)
    model_run_dict[next_key]['delta_t'] = delta_t
    model_run_dict[next_key]['transformations'] = None if transformations== None else ([tf[0].__name__ for tf in transformations], trafo_params)
    model_run_dict[next_key]['comment'] = comment

    return next_key, model_run_dict

## Technology Substitution

In [13]:
bounds =[(0,20), (0,20)]# [(0, np.inf), (0, np.inf)] #
trafo_params = None # [3,3]
transformations =None #  [fractional_transformation(trafo_params[0]), fractional_transformation(trafo_params[0])] #
model = TechnologySubstitution()
num_solver = TechSubNumericalSolver(model)
controls = np.array([0.5, 1])
resolution = [10, 10]
num_samples_per_cell = 100
num_steps = 1
delta_t = 0.5

next_key, run_dict['TechSub'] = update_run_dict(run_dict['TechSub'],num_samples_per_cell, resolution, controls, transformations, trafo_params, delta_t)

In [14]:
for control in controls:
    simulator = run_and_store_simulations(OUTPUT_DIR, 
                          bounds, 
                          transformations, 
                          model, 
                          num_solver, 
                          control, 
                          resolution,
                          num_samples_per_cell,
                          num_steps,
                          delta_t)
    run_dict['TechSub'][next_key]['run_ids'].append(simulator.configs['run_id'][0])

with open(OUTPUT_DIR / "run_dict.json", "w") as file:
    json.dump(run_dict, file, indent=4)

Simulation complete:
- 10000 samples × 1 timesteps = 10000 total rows
- State dimensions: 2
- Control dimensions: 1
Saved results and config.
Stored 10000 rows of simulation data in /home/kstiller/masterthesis/Coding/mdp-world-model/datasets/results/simulation_results.db

Configs data:
            run_id                                           metadata
0  20250327_150530  {"configurations": {"grid": {"bounds": [[0, 20...

Results data:
            run_id trajectory_id   t0   t1        x0        x1   c0        y0  \
0  20250327_150530         0-0_0  0.0  0.5  0.056141  0.451564  0.5  0.056163   
1  20250327_150530         0-0_1  0.0  0.5  1.516251  1.226491  0.5  1.528382   
2  20250327_150530         0-0_2  0.0  0.5  1.561466  0.936346  0.5  1.583662   
3  20250327_150530         0-0_3  0.0  0.5  0.588755  1.423553  0.5  0.589606   
4  20250327_150530         0-0_4  0.0  0.5  0.013150  1.225446  0.5  0.013150   

         y1  
0  0.667044  
1  1.522012  
2  1.200647  
3  1.740932  
4

## FitzHugh-Nagumo

In [62]:
trafo_params = [{'k': 1, 'x_0': 0}, {'k': 1, 'x_0': 0}]
fhntransformations = [logistic_transformation(trafo_params[0]), logistic_transformation(trafo_params[1])]
fhnbounds = [(-np.inf,np.inf),(-np.inf,np.inf)]
fhnmodel = FitzHughNagumoModel(control_params=['b','I'])
fhnsolver = GeneralODENumericalSolver(fhnmodel)
fhncontrols = np.array([[0.5, 0.35], [2, 0], [2, 0.35]])
resolution = [30,30]
num_samples_per_cell = 200
num_steps = 1
delta_t = 0.11

next_key = update_run_dict(run_dict['FHN-Model'],num_samples_per_cell, resolution, fhncontrols, fhntransformations, trafo_params, delta_t)


In [63]:
for control in fhncontrols:
    simulator = run_and_store_simulations(OUTPUT_DIR, 
                          fhnbounds, 
                          fhntransformations, 
                          fhnmodel, 
                          fhnsolver, 
                          control, 
                          resolution,
                          num_samples_per_cell,
                          num_steps,
                          delta_t)
    run_dict['FHN-Model'][next_key]['run_ids'].append(simulator.configs['run_id'][0])
    time.sleep(1) 

with open(OUTPUT_DIR / "run_dict.json", "w") as file:
    json.dump(run_dict, file, indent=4)

Simulation complete:
- 180000 samples × 1 timesteps = 180000 total rows
- State dimensions: 2
- Control dimensions: 2
Saved results and config.
Stored 180000 rows of simulation data in /home/kstiller/masterthesis/Coding/mdp-world-model/datasets/results/simulation_results.db

Configs data:
            run_id                                           metadata
0  20250327_110725  {"configurations": {"grid": {"bounds": [[-Infi...

Results data:
            run_id trajectory_id   t0    t1        x0        x1   c0    c1  \
0  20250327_110725         0-0_0  0.0  0.11 -4.778138 -4.143454  0.5  0.35   
1  20250327_110725         0-0_1  0.0  0.11 -5.288923 -3.715786  0.5  0.35   
2  20250327_110725         0-0_2  0.0  0.11 -5.161365 -5.394979  0.5  0.35   
3  20250327_110725         0-0_3  0.0  0.11 -4.198876 -6.838408  0.5  0.35   
4  20250327_110725         0-0_4  0.0  0.11 -5.236317 -3.525012  0.5  0.35   

         y0        y1  
0 -2.851506 -4.150735  
1 -2.999804 -3.727143  
2 -2.862122 -5